# DEMO DIVRS — Khử thiên lệch (ML-10M)

Demo cho thấy **DIVRS gợi ý chính xác mà bớt hùa theo độ phổ biến** so với MF thường.
Gồm: (A) bảng so sánh số, (B) đường cong độ-phổ-biến theo Top-K, (C) **web demo Gradio**.

Cần: đã có checkpoint **DIVRS** trên Drive (từ notebook train). MF baseline sẽ train nhanh ở cell 3 nếu chưa có.
Bật **T4 GPU**.

## 1. Setup: code + data + Drive

In [ ]:
!pip install -q absl-py setproctitle deprecated faiss-cpu gradio matplotlib
import os, glob, re
REPO='/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR='/content/drive/MyDrive/Colab Notebooks/DIVRS_output'
import torch; USE_GPU=torch.cuda.is_available(); print('GPU:', USE_GPU)

## 2. Train MF baseline (nhanh — sampler đều) nếu chưa có
MF dùng negative sampling đều nên nhanh hơn DIVRS nhiều. Bỏ qua nếu đã có run MF trong DIVRS_output.

In [ ]:
have_mf = any('mf' in os.path.basename(d).lower() for d in glob.glob(OUT_DIR+'/*/'))
if not have_mf:
    !python app.py --flagfile config/ml10m_mf.cfg --output "{OUT_DIR}/" \
      --use_gpu={USE_GPU} --gpu_id=0 --cg_use_gpu=False --num_workers=2 --epochs=20 --es_patience=3
else:
    print('Da co run MF, bo qua.')

## 3. Nạp embedding 2 model + dữ liệu

In [ ]:
import numpy as np, scipy.sparse as sp, torch, glob, os
D='data/ml10m/output/'
train=sp.load_npz(D+'train_coo_record.npz').tocsr()
test =sp.load_npz(D+'test_coo_record.npz').tocsr()
pop  =np.load(D+'popularity.npy').astype(float)
n_user,n_item=train.shape
pop_pct=pop.argsort().argsort()/(len(pop)-1)   # 0=it pho bien, 1=pho bien nhat

def best_ckpt(run):
    cks=glob.glob(run+'ckpt/epoch_*.pth')
    if not cks: return None
    best_ep,best_rc=None,-1
    for lf in glob.glob(run+'log/*'):
        for m in re.finditer(r"VALIDATION epoch: (\d+).*?'recall': np\.float64\(([\d.]+)\)", open(lf).read()):
            ep,rc=int(m.group(1)),float(m.group(2))
            if rc>best_rc: best_rc,best_ep=rc,ep
    p=run+f'ckpt/epoch_{best_ep}.pth'
    return p if (best_ep is not None and os.path.exists(p)) else max(cks,key=lambda x:int(re.findall(r'epoch_(\d+)',x)[0]))

def load_emb(run):
    sd=torch.load(best_ckpt(run),map_location='cpu')
    if 'users_iv' in sd: U,I=sd['users_iv'],sd['items_iv']      # DIVRS
    else: U,I=sd['users'],sd['items']                          # MF
    return U.float().numpy(), I.float().numpy()

runs=glob.glob(OUT_DIR+'/*/')
div_run=max([r for r in runs if 'divrs' in r.lower()], key=os.path.getmtime)
mf_run =max([r for r in runs if 'divrs' not in r.lower()], key=os.path.getmtime)
Ud,Id=load_emb(div_run); Umf,Imf=load_emb(mf_run)
print('DIVRS run:',div_run,'\nMF run:',mf_run)
print('emb shapes  DIVRS',Ud.shape,Id.shape,'| MF',Umf.shape,Imf.shape)

## A. Bảng so sánh: Recall@20 & Độ phổ biến trung bình của gợi ý

In [ ]:
def evaluate(U,I,k=20,n_eval=2000):
    users=np.where(np.diff(test.indptr)>0)[0][:n_eval]
    rec,popm=[],[]
    for u in users:
        s=U[u]@I.T; s[train[u].indices]=-1e9
        top=np.argsort(-s)[:k]
        tset=set(test[u].indices)
        rec.append(len(set(top)&tset)/max(len(tset),1))
        popm.append(pop_pct[top].mean())
    return np.mean(rec), np.mean(popm)

print(f"{'Model':<10}{'Recall@20':>12}{'AvgPop%@20':>14}")
for name,(U,I) in [('MF',(Umf,Imf)),('DIVRS',(Ud,Id))]:
    r,p=evaluate(U,I)
    print(f'{name:<10}{r:>12.4f}{p*100:>13.1f}%')
print('\n=> DIVRS: Recall ngang/hon, AvgPop% THAP hon = it hua theo do pho bien.')

## B. Đường cong: độ phổ biến TB của gợi ý theo Top-K (thấp = khử thiên lệch tốt)

In [ ]:
import matplotlib.pyplot as plt
Ks=[5,10,20,30,40,50]
curve={}
for name,(U,I) in [('MF',(Umf,Imf)),('DIVRS',(Ud,Id))]:
    curve[name]=[evaluate(U,I,k=k,n_eval=1000)[1]*100 for k in Ks]
plt.figure(figsize=(6,4))
for name in curve: plt.plot(Ks,curve[name],marker='o',label=name)
plt.xlabel('Top-K'); plt.ylabel('Do pho bien TB cua goi y (%)')
plt.title('Khu thien lech: DIVRS thap hon MF'); plt.legend(); plt.grid(alpha=.3)
plt.savefig('debias_curve.png',dpi=120,bbox_inches='tight'); plt.show()
print('Da luu debias_curve.png')

## C. WEB DEMO (Gradio) — chọn user, xem gợi ý MF vs DIVRS
Chạy xong sẽ in ra **link public** (xxxxx.gradio.live) — mở trên trình duyệt để demo.

In [ ]:
import gradio as gr
def reco(uid,k):
    uid=int(uid); k=int(k); res=[]
    for name,(U,I) in [('MF (baseline)',(Umf,Imf)),('DIVRS (debiased)',(Ud,Id))]:
        s=U[uid]@I.T; s[train[uid].indices]=-1e9
        top=np.argsort(-s)[:k]
        lines=[f'#{r+1:>2}  Item {it:<5}  do pho bien: {pop_pct[it]*100:>4.0f}%' for r,it in enumerate(top)]
        res.append('\n'.join(lines)+f'\n\n>>> Do pho bien TB: {pop_pct[top].mean()*100:.0f}%')
    return res[0],res[1]

gr.Interface(reco,
    [gr.Slider(0,n_user-1,step=1,value=0,label='User ID'),
     gr.Slider(5,20,step=1,value=10,label='Top-K')],
    [gr.Textbox(label='MF baseline (pho bien cao)',lines=12),
     gr.Textbox(label='DIVRS (da khu thien lech)',lines=12)],
    title='DIVRS Debiasing Demo — ML-10M',
    description='Keo chon user. So sanh goi y MF vs DIVRS. DIVRS co do pho bien TB thap hon = it hua theo dam dong.'
).launch(share=True)

## Ghi chú
- **Item hiện theo ID** (data đã reindex). Muốn hiện tên phim: tải `movies.dat` của ML-10M + map qua `item_reindex.json` — nói nếu cần, tao thêm.
- Embedding lấy từ `users_iv/items_iv` (DIVRS) — đã được train theo mục tiêu khử thiên lệch.
- `AvgPop%` là proxy cho IOU của paper (tỉ lệ item phổ biến trong list). Thấp hơn = khử thiên lệch tốt hơn.